# финальная модель задачи 1: гетерогенный бленд

это седьмой ноутбук конвейера, поверх `04_task1_rating_mlp`. сеть InteractionMLP из 04 - сильная одиночная модель, но её ошибки можно ещё уменьшить, если усреднить несколько *разных по индуктивному сдвигу* моделей: их промахи частично декоррелируют. здесь мы собираем такой бленд и подбираем веса членов строго по val.

финальная модель - взвешенная смесь пяти членов, **test RMSE 1.0424** против **1.0449** у baseline-ансамбля из ноутбука 04 (тот же срез и протокол, единое окружение), выигрыш равномерный по всем сегментам, включая холодный старт. весь код членов и бленда лежит в `scripts/task1_*.py`, отбор по val, test смотрим один раз. утечек не добавляли - та же feature-схема, что у baseline.

## состав бленда

| член | что это | вес |
|---|---|---|
| coral | та же сеть с порядковой головой CORAL вместо softmax-матожидания, 3 сида | 0.36 |
| gbdt | HistGradientBoostingRegressor на тех же признаках (другой класс модели) | 0.19 |
| recency | та же сеть с recency-взвешиванием обучения (свежие отзывы весомее) | 0.18 |
| ens_ext | та же сеть + обогащённые prior-признаки (разброс и доли крайних оценок) | 0.18 |
| ens | baseline-ансамбль из ноутбука 04 | 0.10 |

веса подобраны жадным отбором на val (стэкинг). главную диверсификацию дают gbdt и порядковая голова: дерево ошибается не там, где сеть, а CORAL иначе кодирует порядок оценок. одиночные члены и абляции, которые в бленд не вошли (например recency с сильным весом и обогащённые priors сами по себе), разобраны в `scripts/task1_experiments.py`.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root)); sys.path.insert(0, str(root / "scripts"))
from _constants import REPORTS

b = json.load(open(REPORTS / "task1_model_metrics.json", encoding="utf-8"))["blend"]
print("baseline-ансамбль (тот же срез, единое окружение):", round(b["baseline_ensemble_test_rmse_same_env"], 4))
print("финальный бленд:", round(b["test_rmse"], 4))
pd.DataFrame({
    "член": list(b["base_test_rmse"]),
    "test RMSE одиночно": [round(v, 4) for v in b["base_test_rmse"].values()],
    "вес в бленде": [b["weights"].get(k, 0.0) for k in b["base_test_rmse"]],
})

одиночно члены примерно на уровне baseline (gbdt слабее, это нормально для деревьев на наших признаках), но вместе они снижают RMSE: бленд ниже любого своего члена. ниже - по сегментам теста.

In [ ]:
pd.DataFrame([{"сегмент": k, "бленд RMSE": round(v, 4)} for k, v in b["segments"].items()])

бленд выигрывает у baseline на каждом сегменте, включая холодных юзеров и холодные заведения, - значит выигрыш не куплен ценой какого-то среза, а равномерный.

## восстановимость

финальная модель сохранена как единый артефакт `logs/task1/final/blend.pt` (веса всех членов, конфиг архитектуры и веса бленда) плюс `logs/task1/final/gbdt.joblib`. `scripts/task1_blend_infer.py` восстанавливает её и считает предсказание без переобучения и без чтения ноутбука - ниже сверяем, что артефакт даёт тот же RMSE.

In [ ]:
from task1_blend_infer import predict_blend, interpretable_metrics
import task1_lib as L

pred, y, D = predict_blend("test")
print("из артефакта test RMSE:", round(L.rmse(y, pred), 4))
im = interpretable_metrics(pred, y)
print(f"точное попадание в звезду: {im['exact_hit']:.1%}")
print(f"ошибка не больше 1 звезды: {im['within_1_star']:.1%}")
print(f"бинарно понравится (порог 4): {im['binary_acc_thr4']:.1%} против наивного {im['binary_naive']:.1%}")

## вывод

бленд снижает test RMSE с 1.0449 до 1.0424 (-0.0025), выигрыш согласован между val и test и держится на всех сегментах. главный вклад дают порядковая голова CORAL и GBDT - они ошибаются иначе, чем baseline-сеть, поэтому усреднение их с baseline уменьшает дисперсию ошибки. recency-взвешивание само по себе пользы не дало (оптимум - почти равные веса, то есть исходный baseline), но как слабо-диверсный член в бленде вес получает. подробный разбор всех членов, абляций и негативных результатов - в `reports/task1_model_metrics.json` и `scripts/`, обучение финальной модели - в `scripts/task1_finalize.py`.